# Module 3 — Question Answering
**Pipeline:** IR → Summarization → **QA**  
**Datasets:** SQuAD 2.0 (single-hop) · HotpotQA (multi-hop)  
**Model:** BiLSTM + Attention span extractor (built from scratch with PyTorch)  
**Embeddings:** GloVe 100d (pre-trained, no fine-tuning)  
**Evaluation:** Exact Match (EM) + F1

## 1. Install & Import

In [1]:
!pip install datasets -q

import os, re, json, string, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings"
DEVICE = torch.device('cuda')
print(f'Device: {DEVICE}')

Device: cuda


## 2. Load GloVe Embeddings

In [2]:
# Download GloVe 100d (822 MB) — only needed once per Kaggle session
GLOVE_PATH = '/kaggle/working/glove.6B.100d.txt'

if not os.path.exists(GLOVE_PATH):
    print('Downloading GloVe...')
    os.system('wget -q https://nlp.stanford.edu/data/glove.6B.zip -O /kaggle/working/glove.6B.zip')
    os.system('unzip -q /kaggle/working/glove.6B.zip -d /kaggle/working/')
    print('Done.')
else:
    print('GloVe already downloaded.')

def load_glove(path, max_words=None):
    print('Loading GloVe vectors...')
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    vectors  = [np.zeros(100), np.random.randn(100) * 0.01]
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_words and i >= max_words:
                break
            parts = line.split()
            word  = parts[0]
            vec   = np.array(parts[1:], dtype=np.float32)
            word2idx[word] = len(word2idx)
            vectors.append(vec)
    embeddings = np.stack(vectors).astype(np.float32)
    print(f'Loaded {len(word2idx)} words, embedding dim={embeddings.shape[1]}')
    return word2idx, embeddings

WORD2IDX, EMBEDDINGS = load_glove(GLOVE_PATH, max_words=200_000)

Done.
Loading GloVe vectors...
Loaded 200002 words, embedding dim=100


## 3. Tokenizer & Vocabulary

In [4]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

def encode(tokens, word2idx, max_len):
    ids = [word2idx.get(t, 1) for t in tokens[:max_len]]
    pad = max_len - len(ids)
    return ids + [0] * pad, len(ids)

# Constants
MAX_CTX = 400    # max tokens for context/passage
MAX_Q   = 50     # max tokens for question
EMB_DIM = 100
HIDDEN  = 128
BATCH   = 32

## 4. Load & Preprocess SQuAD 2.0

In [5]:
print('Loading SQuAD 2.0...')
squad = load_dataset('rajpurkar/squad_v2')

def find_token_span(ctx_tokens, answer_text, answer_start_char):
    """Find token-level start/end indices for the answer."""
    ctx_lower = ' '.join(ctx_tokens)
    # rough heuristic: find character offsets
    char_idx = 0
    token_starts = []
    reconstructed = ''
    for t in ctx_tokens:
        token_starts.append(char_idx)
        char_idx += len(t) + 1  # +1 for space

    ans_tokens = tokenize(answer_text)
    # Search for the answer token sequence in context
    for i in range(len(ctx_tokens) - len(ans_tokens) + 1):
        if ctx_tokens[i:i + len(ans_tokens)] == ans_tokens:
            return i, i + len(ans_tokens) - 1
    return None, None

def process_squad(split, max_samples=10_000):
    data = []
    for item in list(squad[split])[:max_samples]:
        ctx_tokens = tokenize(item['context'])
        q_tokens   = tokenize(item['question'])
        answerable = len(item['answers']['text']) > 0

        if answerable:
            ans_text  = item['answers']['text'][0]
            ans_start = item['answers']['answer_start'][0]
            s, e = find_token_span(ctx_tokens, ans_text, ans_start)
            if s is None or s >= MAX_CTX:
                continue
            e = min(e, MAX_CTX - 1)
        else:
            s, e = 0, 0    # unanswerable — will use answerable flag

        ctx_ids,  ctx_len = encode(ctx_tokens,  WORD2IDX, MAX_CTX)
        q_ids,    q_len   = encode(q_tokens,    WORD2IDX, MAX_Q)

        data.append({
            'ctx_ids':    ctx_ids,
            'ctx_len':    ctx_len,
            'q_ids':      q_ids,
            'q_len':      q_len,
            'start':      s,
            'end':        e,
            'answerable': int(answerable),
            'context':    item['context'],
            'question':   item['question'],
            'answer':     item['answers']['text'][0] if answerable else '',
            'ctx_tokens': ctx_tokens,
        })
    return data

print('Processing train split...')
train_raw = process_squad('train', max_samples=20_000)
print('Processing validation split...')
val_raw   = process_squad('validation', max_samples=3_000)
print(f'Train: {len(train_raw)} | Val: {len(val_raw)}')

Loading SQuAD 2.0...


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Processing train split...
Processing validation split...
Train: 19902 | Val: 2998


## 5. PyTorch Dataset

In [6]:

class QADataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return (
            torch.tensor(item['ctx_ids'],    dtype=torch.long),
            torch.tensor(item['q_ids'],      dtype=torch.long),
            torch.tensor(item['ctx_len'],    dtype=torch.long),
            torch.tensor(item['q_len'],      dtype=torch.long),
            torch.tensor(item['start'],      dtype=torch.long),
            torch.tensor(item['end'],        dtype=torch.long),
            torch.tensor(item['answerable'], dtype=torch.float),
        )

train_ds = QADataset(train_raw)
val_ds   = QADataset(val_raw)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 622 | Val batches: 94


## 6. BiLSTM + Attention QA Model

In [7]:
class BidafAttention(nn.Module):
    """Simplified Bidirectional Attention Flow between context and query."""

    def __init__(self, hidden_size):
        super().__init__()
        self.w_ctx   = nn.Linear(hidden_size * 2, 1, bias=False)
        self.w_q     = nn.Linear(hidden_size * 2, 1, bias=False)
        self.w_cq    = nn.Linear(hidden_size * 2, 1, bias=False)

    def forward(self, ctx, q, ctx_mask, q_mask):
        # ctx  : (B, T, 2H)  q : (B, J, 2H)
        B, T, H2 = ctx.shape
        J = q.shape[1]

        ctx_exp = ctx.unsqueeze(2).expand(B, T, J, H2)   # (B,T,J,2H)
        q_exp   = q.unsqueeze(1).expand(B, T, J, H2)     # (B,T,J,2H)

        s = (self.w_ctx(ctx_exp) + self.w_q(q_exp) +
             self.w_cq(ctx_exp * q_exp)).squeeze(-1)      # (B,T,J)

        # Mask padding
        if q_mask is not None:
            s = s.masked_fill(~q_mask.unsqueeze(1), -1e9)

        # Context-to-Query attention
        a = F.softmax(s, dim=2)                           # (B,T,J)
        A = torch.bmm(a, q)                               # (B,T,2H)

        # Query-to-Context attention
        b = F.softmax(s.max(dim=2).values, dim=1)         # (B,T)
        if ctx_mask is not None:
            b = b.masked_fill(~ctx_mask, 1e-9)
            b = b / (b.sum(dim=1, keepdim=True) + 1e-9)
        B_vec = torch.bmm(b.unsqueeze(1), ctx).expand_as(ctx)  # (B,T,2H)

        G = torch.cat([ctx, A, ctx * A, ctx * B_vec], dim=-1)  # (B,T,8H)
        return G


class QAModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, pretrained_embs, dropout=0.3):
        super().__init__()

        # Embedding layer (frozen GloVe)
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(pretrained_embs))
        self.embedding.weight.requires_grad = True

        self.emb_proj = nn.Linear(emb_dim, hidden)
        self.dropout  = nn.Dropout(dropout)

        # Encoders
        self.ctx_encoder = nn.LSTM(hidden, hidden, batch_first=True,
                                   bidirectional=True, num_layers=2, dropout=dropout)
        self.q_encoder   = nn.LSTM(hidden, hidden, batch_first=True,
                                   bidirectional=True, num_layers=2, dropout=dropout)

        # Attention
        self.attention = BidafAttention(hidden)

        # Modeling layer
        self.model_lstm = nn.LSTM(hidden * 8, hidden, batch_first=True,
                                  bidirectional=True, num_layers=2, dropout=dropout)

        # Output: start / end logits
        self.span_start = nn.Linear(hidden * 10, 1)
        self.span_end   = nn.Linear(hidden * 10, 1)

        # Answerability classifier
        self.answerable = nn.Linear(hidden * 2, 1)

    def make_mask(self, lengths, max_len):
        bsz = lengths.size(0)
        mask = torch.arange(max_len, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)
        return mask

    def forward(self, ctx_ids, q_ids, ctx_lens, q_lens):
        # Masks
        ctx_mask = self.make_mask(ctx_lens, ctx_ids.size(1))
        q_mask   = self.make_mask(q_lens,   q_ids.size(1))

        # Embeddings
        ctx_emb = self.dropout(F.relu(self.emb_proj(self.embedding(ctx_ids))))  # (B,T,H)
        q_emb   = self.dropout(F.relu(self.emb_proj(self.embedding(q_ids))))    # (B,J,H)

        # Encode
        ctx_enc, _ = self.ctx_encoder(ctx_emb)   # (B,T,2H)
        q_enc,   _ = self.q_encoder(q_emb)        # (B,J,2H)

        # BiDAF attention
        G = self.attention(ctx_enc, q_enc, ctx_mask, q_mask)   # (B,T,8H)
        M, _ = self.model_lstm(G)                               # (B,T,2H)

        cat = torch.cat([G, M], dim=-1)                         # (B,T,10H)

        start_logits = self.span_start(cat).squeeze(-1)         # (B,T)
        end_logits   = self.span_end(cat).squeeze(-1)           # (B,T)

        # Mask padding
        start_logits = start_logits.masked_fill(~ctx_mask, -1e9)
        end_logits   = end_logits.masked_fill(~ctx_mask,   -1e9)

        # Answerability: pool M over valid positions
        m_pool = (M * ctx_mask.unsqueeze(-1).float()).sum(1) / ctx_lens.unsqueeze(-1).float()
        ans_logit = self.answerable(m_pool).squeeze(-1)         # (B,)

        return start_logits, end_logits, ans_logit


# Instantiate model
model = QAModel(
    vocab_size=len(WORD2IDX),
    emb_dim=EMB_DIM,
    hidden=HIDDEN,
    pretrained_embs=EMBEDDINGS,
    dropout=0.4,      # was 0.3
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

Trainable parameters: 22,912,587


## 7. Loss & Optimizer

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

def compute_loss(start_logits, end_logits, ans_logits, starts, ends, answerables):
    ans_loss = F.binary_cross_entropy_with_logits(ans_logits, answerables)
    
    # Only compute span loss on answerable samples
    mask = answerables.bool()
    if mask.sum() == 0:
        return ans_loss
    
    span_loss = (
        F.cross_entropy(start_logits[mask], starts[mask]) +
        F.cross_entropy(end_logits[mask],   ends[mask])
    ) / 2
    return span_loss + 0.5 * ans_loss

## 8. Training Loop

In [9]:
def normalize_answer(s):
    def remove_articles(t): return re.sub(r'\b(a|an|the)\b', ' ', t)
    def white_space_fix(t): return ' '.join(t.split())
    def remove_punc(t): return ''.join(c for c in t if c not in set(string.punctuation))
    def lower(t): return t.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def get_tokens(s):
    return normalize_answer(s).split()

def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
    gold_toks = get_tokens(a_gold)
    pred_toks = get_tokens(a_pred)
    common    = collections.Counter(gold_toks) & collections.Counter(pred_toks)
    num_same  = sum(common.values())
    if not gold_toks or not pred_toks or not num_same:
        return 0.0
    precision = num_same / len(pred_toks)
    recall    = num_same / len(gold_toks)
    return (2 * precision * recall) / (precision + recall)

def predict_span(model, item):
    model.eval()
    ctx_ids, ctx_len = encode(item['ctx_tokens'], WORD2IDX, MAX_CTX)
    q_tokens = tokenize(item['question'])
    q_ids, q_len     = encode(q_tokens, WORD2IDX, MAX_Q)

    ctx_t = torch.tensor([ctx_ids], dtype=torch.long).to(DEVICE)
    q_t   = torch.tensor([q_ids],   dtype=torch.long).to(DEVICE)
    cl_t  = torch.tensor([ctx_len], dtype=torch.long).to(DEVICE)
    ql_t  = torch.tensor([q_len],   dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        s_log, e_log, a_log = model(ctx_t, q_t, cl_t, ql_t)

    answerable = torch.sigmoid(a_log).item() > 0.3  # was 0.5
    if not answerable:
        return ''

    # Enforce start <= end with a 2D grid trick
    s_probs = F.softmax(s_log[0], dim=0).cpu().numpy()
    e_probs = F.softmax(e_log[0], dim=0).cpu().numpy()
    best_score, best_s, best_e = -1, 0, 0
    for s in range(min(ctx_len, MAX_CTX)):
        for e in range(s, min(s + 15, ctx_len, MAX_CTX)):
            score = s_probs[s] * e_probs[e]
            if score > best_score:
                best_score, best_s, best_e = score, s, e

    return ' '.join(item['ctx_tokens'][best_s:best_e + 1])



In [10]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for ctx_ids, q_ids, ctx_lens, q_lens, starts, ends, ans in loader:
        ctx_ids  = ctx_ids.to(DEVICE)
        q_ids    = q_ids.to(DEVICE)
        ctx_lens = ctx_lens.to(DEVICE)
        q_lens   = q_lens.to(DEVICE)
        starts   = starts.to(DEVICE)
        ends     = ends.to(DEVICE)
        ans      = ans.to(DEVICE)

        optimizer.zero_grad()
        s_logits, e_logits, a_logits = model(ctx_ids, q_ids, ctx_lens, q_lens)
        loss = compute_loss(s_logits, e_logits, a_logits, starts, ends, ans)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    for ctx_ids, q_ids, ctx_lens, q_lens, starts, ends, ans in loader:
        ctx_ids  = ctx_ids.to(DEVICE); q_ids = q_ids.to(DEVICE)
        ctx_lens = ctx_lens.to(DEVICE); q_lens = q_lens.to(DEVICE)
        starts   = starts.to(DEVICE);  ends   = ends.to(DEVICE)
        ans      = ans.to(DEVICE)
        s_logits, e_logits, a_logits = model(ctx_ids, q_ids, ctx_lens, q_lens)
        loss = compute_loss(s_logits, e_logits, a_logits, starts, ends, ans)
        total_loss += loss.item()
    return total_loss / len(loader)

# --- Train ---
EPOCHS = 30
best_val_f1 = -1.0
patience = 3         # stop after 3 epochs of no F1 improvement
no_improve = 0
os.makedirs('/kaggle/working/qa_artifacts', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    tr_loss  = train_epoch(model, train_loader, optimizer)
    if epoch == 1:
        model.embedding.weight.requires_grad = True
        print('  Embeddings unfrozen.')
    val_loss = eval_epoch(model, val_loader)

    # Compute EM + F1 on first 200 val samples
    em_ep, f1_ep = [], []
    model.eval()
    answerable_val = [x for x in val_raw if x['answer']]
    for item in answerable_val[:600]:
        pred = predict_span(model, item)
        em_ep.append(compute_exact(item['answer'], pred))
        f1_ep.append(compute_f1(item['answer'], pred))

    val_em = np.mean(em_ep) if em_ep else 0.0
    val_f1 = np.mean(f1_ep) if f1_ep else 0.0
    scheduler.step(val_f1)  # step on F1, not loss

    print(f'Epoch {epoch}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val EM: {val_em:.4f} | Val F1: {val_f1:.4f}')

    if val_f1 > best_val_f1:         # ← save on F1, not loss
        best_val_f1 = val_f1
        torch.save(model.state_dict(), '/kaggle/working/qa_best.pt')
        print('  ✓ Saved best model.')
        no_improve = 0
    else : 
        no_improve += 1
        if no_improve >= patience:
            print(f'Early stop at epoch {epoch}.')
            break
        

  Embeddings unfrozen.
Epoch 1/30 | Train Loss: 4.1406 | Val Loss: 4.2104 | Val EM: 0.0550 | Val F1: 0.1233
  ✓ Saved best model.
Epoch 2/30 | Train Loss: 3.7606 | Val Loss: 4.1452 | Val EM: 0.0550 | Val F1: 0.1241
  ✓ Saved best model.
Epoch 3/30 | Train Loss: 3.6187 | Val Loss: 4.1171 | Val EM: 0.0533 | Val F1: 0.1180
Epoch 4/30 | Train Loss: 3.5014 | Val Loss: 4.1087 | Val EM: 0.0633 | Val F1: 0.1371
  ✓ Saved best model.
Epoch 5/30 | Train Loss: 3.3067 | Val Loss: 4.0420 | Val EM: 0.0867 | Val F1: 0.1543
  ✓ Saved best model.
Epoch 6/30 | Train Loss: 3.1410 | Val Loss: 4.1214 | Val EM: 0.0900 | Val F1: 0.1532
Epoch 7/30 | Train Loss: 2.9828 | Val Loss: 4.2148 | Val EM: 0.1017 | Val F1: 0.1686
  ✓ Saved best model.
Epoch 8/30 | Train Loss: 2.9062 | Val Loss: 4.1605 | Val EM: 0.0883 | Val F1: 0.1561
Epoch 9/30 | Train Loss: 2.8381 | Val Loss: 4.1829 | Val EM: 0.0850 | Val F1: 0.1557
Epoch 10/30 | Train Loss: 2.7444 | Val Loss: 4.2176 | Val EM: 0.1050 | Val F1: 0.1683
Early stop at ep

## 9. EM & F1 Evaluation

In [11]:
# Load best model
model.load_state_dict(torch.load('/kaggle/working/qa_best.pt'))

# Evaluate EM + F1
print('Evaluating EM + F1 on validation set...')
em_scores, f1_scores = [], []
for item in val_raw[:500]:
    if not item['answer']:
        continue
    pred = predict_span(model, item)
    em_scores.append(compute_exact(item['answer'], pred))
    f1_scores.append(compute_f1(item['answer'], pred))

print(f'Exact Match : {np.mean(em_scores):.4f}')
print(f'F1 Score    : {np.mean(f1_scores):.4f}')

Evaluating EM + F1 on validation set...
Exact Match : 0.0844
F1 Score    : 0.1518


## 10. HotpotQA — Multi-Hop QA Evaluation

In [12]:
print('Loading HotpotQA...')
hotpot = load_dataset('hotpot_qa', 'distractor')  # distractor = realistic multi-hop

def process_hotpot(split, max_samples=2_000):
    data = []
    for item in list(hotpot[split])[:max_samples]:
        # Concatenate all supporting paragraphs as context
        context_parts = []
        for title, sentences in zip(item['context']['title'], item['context']['sentences']):
            context_parts.append(' '.join(sentences))
        full_ctx = ' '.join(context_parts)

        ctx_tokens = tokenize(full_ctx)
        q_tokens   = tokenize(item['question'])
        ans_text   = item['answer']

        s, e = find_token_span(ctx_tokens, ans_text, 0)
        if s is None or s >= MAX_CTX:
            continue
        e = min(e, MAX_CTX - 1)

        ctx_ids, ctx_len = encode(ctx_tokens, WORD2IDX, MAX_CTX)
        q_ids,   q_len   = encode(q_tokens,   WORD2IDX, MAX_Q)

        data.append({
            'ctx_ids': ctx_ids, 'ctx_len': ctx_len,
            'q_ids': q_ids,     'q_len': q_len,
            'start': s, 'end': e, 'answerable': 1,
            'context': full_ctx, 'question': item['question'],
            'answer': ans_text,  'ctx_tokens': ctx_tokens,
        })
    return data

hotpot_val = process_hotpot('validation', max_samples=1_000)
print(f'HotpotQA val processed: {len(hotpot_val)}')

# Evaluate on HotpotQA (zero-shot transfer from SQuAD)
em_hot, f1_hot = [], []
for item in hotpot_val[:300]:
    pred = predict_span(model, item)
    em_hot.append(compute_exact(item['answer'], pred))
    f1_hot.append(compute_f1(item['answer'], pred))

print(f'\nHotpotQA (multi-hop) — zero-shot transfer:')
print(f'  Exact Match : {np.mean(em_hot):.4f}')
print(f'  F1 Score    : {np.mean(f1_hot):.4f}')

Loading HotpotQA...


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

HotpotQA val processed: 497

HotpotQA (multi-hop) — zero-shot transfer:
  Exact Match : 0.0400
  F1 Score    : 0.0655


## 11. Full Pipeline Demo — IR → Summarization → QA

In [14]:
!pip install rank_bm25 -q

In [15]:
# ---- Inline minimal IR + Summarization so this notebook is self-contained ----
import pickle, scipy.sparse as sp
from rank_bm25 import BM25Okapi

# Load IR artifacts saved in Module 1
IR_PATH = '/kaggle/working/ir_artifacts'

try:
    with open(f'{IR_PATH}/tfidf_vectorizer.pkl', 'rb') as f:
        tfidf_vec = pickle.load(f)
    tfidf_mat = sp.load_npz(f'{IR_PATH}/tfidf_matrix.npz')
    with open(f'{IR_PATH}/bm25.pkl', 'rb') as f:
        bm25_model = pickle.load(f)
    with open(f'{IR_PATH}/corpus_index.json') as f:
        corpus = json.load(f)
    corpus_ids      = corpus['ids']
    corpus_passages = corpus['passages']
    IR_AVAILABLE = True
    print('IR artifacts loaded.')
except FileNotFoundError:
    IR_AVAILABLE = False
    print('IR artifacts not found — using dummy retrieval for demo.')


def pipeline_retrieve(question, top_k=5):
    if not IR_AVAILABLE:
        # Fallback: pick passages from val set that contain question words
        q_words = set(question.lower().split())
        results = []
        for item in val_raw[:200]:
            overlap = sum(1 for w in q_words if w in item['context'].lower())
            results.append((overlap, item['context']))
        results.sort(reverse=True)
        return [r[1] for r in results[:top_k]]

    from sklearn.metrics.pairwise import cosine_similarity as cos_sim
    import re
    q_clean = ' '.join(re.findall(r'[a-z]+', question.lower()))
    q_tok   = q_clean.split()

    tf_scores = cos_sim(tfidf_vec.transform([q_clean]), tfidf_mat).flatten()
    bm_scores = np.array(bm25_model.get_scores(q_tok))

    def norm(a): return (a - a.min()) / (a.max() - a.min() + 1e-9)
    combined = 0.5 * norm(tf_scores) + 0.5 * norm(bm_scores)
    top_idx  = np.argsort(combined)[::-1][:top_k]
    return [corpus_passages[i] for i in top_idx]


def simple_extractive_summary(text, max_sents=3):
    """Lightweight summarizer (no dependencies on Module 2 artifacts)."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return ' '.join(sentences[:max_sents])


def full_pipeline(question, top_k=5, verbose=True):
    if verbose:
        print(f'\n{'='*70}')
        print(f'QUESTION: {question}')
        print('='*70)

    # Step 1 — IR
    passages = pipeline_retrieve(question, top_k=top_k)
    if verbose:
        print(f'\n[IR] Retrieved {len(passages)} passages.')

    # Step 2 — Summarization
    summaries = [simple_extractive_summary(p) for p in passages]
    if verbose:
        print('[Summ] Passages compressed.')

    # Step 3 — QA on each summarized passage
    best_answer, best_f1 = '', 0
    for i, summary in enumerate(summaries):
        ctx_tokens = tokenize(summary)
        item = {
            'ctx_tokens': ctx_tokens,
            'question':   question,
            'answer':     '',
            'ctx_ids':    encode(ctx_tokens, WORD2IDX, MAX_CTX)[0],
        }
        pred = predict_span(model, item)
        if pred and len(pred.split()) >= 1:
            best_answer = pred
            if verbose:
                print(f'  Passage {i+1}: "{pred}"')
            break  # take first non-empty answer

    if verbose:
        print(f'\n✅ FINAL ANSWER: {best_answer or "[No answer found]"}')
    return best_answer

# Run demos
full_pipeline('Who invented the telephone?')
full_pipeline('When did World War II end?')
full_pipeline('What is the speed of light?')

IR artifacts not found — using dummy retrieval for demo.

QUESTION: Who invented the telephone?

[IR] Retrieved 5 passages.
[Summ] Passages compressed.
  Passage 1: "norman - educated mind"

✅ FINAL ANSWER: norman - educated mind

QUESTION: When did World War II end?

[IR] Retrieved 5 passages.
[Summ] Passages compressed.
  Passage 1: "when"

✅ FINAL ANSWER: when

QUESTION: What is the speed of light?

[IR] Retrieved 5 passages.
[Summ] Passages compressed.
  Passage 1: "french revolution"

✅ FINAL ANSWER: french revolution


'french revolution'

## 12. Results Summary

In [16]:
print()
print('='*60)
print('         PROJECT RESULTS SUMMARY')
print('='*60)
print()
print('MODULE 1 — Information Retrieval (SQuAD 2.0)')
print('  TF-IDF Recall@10  : see Module 1 notebook')
print('  BM25   Recall@10  : see Module 1 notebook')
print('  Hybrid MRR        : see Module 1 notebook')
print()
print('MODULE 2 — Text Summarization')
print('  CNN/DailyMail ROUGE-1 / ROUGE-2 / ROUGE-L : see Module 2 notebook')
print('  XSum cross-dataset ROUGE : see Module 2 notebook')
print()
print('MODULE 3 — Question Answering')
print(f'  SQuAD 2.0   EM  : {np.mean(em_scores):.4f}')
print(f'  SQuAD 2.0   F1  : {np.mean(f1_scores):.4f}')
print(f'  HotpotQA    EM  : {np.mean(em_hot):.4f}  (zero-shot)')
print(f'  HotpotQA    F1  : {np.mean(f1_hot):.4f}  (zero-shot)')
print()
print('PIPELINE: IR → Summarization → QA  ✅ end-to-end')
print('='*60)


         PROJECT RESULTS SUMMARY

MODULE 1 — Information Retrieval (SQuAD 2.0)
  TF-IDF Recall@10  : see Module 1 notebook
  BM25   Recall@10  : see Module 1 notebook
  Hybrid MRR        : see Module 1 notebook

MODULE 2 — Text Summarization
  CNN/DailyMail ROUGE-1 / ROUGE-2 / ROUGE-L : see Module 2 notebook
  XSum cross-dataset ROUGE : see Module 2 notebook

MODULE 3 — Question Answering
  SQuAD 2.0   EM  : 0.0844
  SQuAD 2.0   F1  : 0.1518
  HotpotQA    EM  : 0.0400  (zero-shot)
  HotpotQA    F1  : 0.0655  (zero-shot)

PIPELINE: IR → Summarization → QA  ✅ end-to-end


In [19]:
ls qa_artifacts